In [0]:
import boto3
import json
import socket
import requests
from botocore.exceptions import ClientError, EndpointConnectionError, NoCredentialsError
from io import StringIO, BytesIO
import pandas as pd

# ── Config ────────────────────────────────────────────────────
MINIO_ENDPOINT   = "https://lej1.mooo.com"
MINIO_ACCESS_KEY = "eujin"
MINIO_SECRET_KEY = "lejin2000"
BUCKET_NAME      = "test-bucket"
TEST_PREFIX      = "connectivity_test"

# ── Helpers ───────────────────────────────────────────────────
def print_header(title: str):
    print(f"\n{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")

def ok(msg):  print(f"  ✅ {msg}")
def fail(msg): print(f"  ❌ {msg}")
def warn(msg): print(f"  ⚠️  {msg}")

# ─────────────────────────────────────────────────────────────
# TEST 1: DNS Resolution
# ─────────────────────────────────────────────────────────────
print_header("TEST 1: DNS Resolution")
try:
    domain = MINIO_ENDPOINT.replace("https://", "").replace("http://", "").split(":")[0]
    ip = socket.gethostbyname(domain)
    ok(f"'{domain}' resolved to {ip}")
except socket.gaierror as e:
    fail(f"DNS resolution failed: {e}")

# ─────────────────────────────────────────────────────────────
# TEST 2: HTTPS Reachability (Caddy health check)
# ─────────────────────────────────────────────────────────────
print_header("TEST 2: HTTPS Reachability via Caddy")
try:
    response = requests.get(f"{MINIO_ENDPOINT}/minio/health/live", timeout=5)
    if response.status_code == 200:
        ok(f"MinIO health endpoint reachable (HTTP {response.status_code})")
    else:
        warn(f"Unexpected status code: {response.status_code}")
except requests.exceptions.SSLError as e:
    fail(f"SSL/TLS error (check Caddy cert): {e}")
except requests.exceptions.ConnectionError as e:
    fail(f"Cannot reach endpoint (check Caddy/firewall): {e}")
except requests.exceptions.Timeout:
    fail("Request timed out — host unreachable or slow")

# ─────────────────────────────────────────────────────────────
# TEST 3: boto3 Client Authentication
# ─────────────────────────────────────────────────────────────
print_header("TEST 3: boto3 Authentication")
try:
    s3_client = boto3.client(
        "s3",
        endpoint_url          = MINIO_ENDPOINT,
        aws_access_key_id     = MINIO_ACCESS_KEY,
        aws_secret_access_key = MINIO_SECRET_KEY,
        region_name           = "us-east-1"
    )
    response = s3_client.list_buckets()
    buckets  = [b["Name"] for b in response.get("Buckets", [])]
    ok(f"Authenticated successfully!")
    ok(f"Existing buckets: {buckets if buckets else '(none)'}")
except NoCredentialsError:
    fail("No credentials provided")
except ClientError as e:
    fail(f"Auth failed: {e.response['Error']['Code']} — {e.response['Error']['Message']}")
except EndpointConnectionError as e:
    fail(f"Cannot connect to endpoint: {e}")
except Exception as e:
    fail(f"Unexpected error: {e}")

# ─────────────────────────────────────────────────────────────
# TEST 4: Bucket Access / Auto-create
# ─────────────────────────────────────────────────────────────
print_header(f"TEST 4: Bucket Access → '{BUCKET_NAME}'")
try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    ok(f"Bucket '{BUCKET_NAME}' exists and is accessible")
except ClientError as e:
    error_code = e.response["Error"]["Code"]
    if error_code in ("404", "NoSuchBucket"):
        warn(f"Bucket '{BUCKET_NAME}' not found — attempting to create...")
        try:
            s3_client.create_bucket(Bucket=BUCKET_NAME)
            ok(f"Bucket '{BUCKET_NAME}' created successfully")
        except Exception as ce:
            fail(f"Could not create bucket: {ce}")
    elif error_code == "403":
        fail("Access denied — check MinIO bucket policy or credentials")
    else:
        fail(f"Bucket error: {error_code}")

# ─────────────────────────────────────────────────────────────
# TEST 5: Write a Text File
# ─────────────────────────────────────────────────────────────
print_header("TEST 5: Write Text File")
TEST_KEY_TXT = f"{TEST_PREFIX}/hello.txt"
try:
    s3_client.put_object(
        Bucket = BUCKET_NAME,
        Key    = TEST_KEY_TXT,
        Body   = "MinIO connection from Databricks — OK ✅"
    )
    ok(f"Text file written → s3://{BUCKET_NAME}/{TEST_KEY_TXT}")
except Exception as e:
    fail(f"Write failed: {e}")

# ─────────────────────────────────────────────────────────────
# TEST 6: Read Back Text File
# ─────────────────────────────────────────────────────────────
print_header("TEST 6: Read Back Text File")
try:
    response = s3_client.get_object(Bucket=BUCKET_NAME, Key=TEST_KEY_TXT)
    content  = response["Body"].read().decode("utf-8")
    ok(f"Read successful! Content: '{content}'")
except Exception as e:
    fail(f"Read failed: {e}")

# ─────────────────────────────────────────────────────────────
# TEST 7: Write a pandas DataFrame as CSV
# ─────────────────────────────────────────────────────────────
print_header("TEST 7: Write pandas DataFrame as CSV")
TEST_KEY_CSV = f"{TEST_PREFIX}/test_dataframe.csv"
try:
    sample_df = pd.DataFrame({
        "ticker":            ["AAPL", "MSFT", "GOOGL"],
        "dividend_per_share": [0.25,   0.75,   0.00],
        "year":              [2024,    2024,   2024]
    })
    csv_buffer = StringIO()
    sample_df.to_csv(csv_buffer, index=False)
    s3_client.put_object(
        Bucket = BUCKET_NAME,
        Key    = TEST_KEY_CSV,
        Body   = csv_buffer.getvalue()
    )
    ok(f"CSV DataFrame written → s3://{BUCKET_NAME}/{TEST_KEY_CSV}")
except Exception as e:
    fail(f"CSV write failed: {e}")

# ─────────────────────────────────────────────────────────────
# TEST 8: Write a pandas DataFrame as Parquet
# ─────────────────────────────────────────────────────────────
print_header("TEST 8: Write pandas DataFrame as Parquet")
TEST_KEY_PARQUET = f"{TEST_PREFIX}/test_dataframe.parquet"
try:
    parquet_buffer = BytesIO()
    sample_df.to_parquet(parquet_buffer, index=False)
    parquet_buffer.seek(0)
    s3_client.put_object(
        Bucket = BUCKET_NAME,
        Key    = TEST_KEY_PARQUET,
        Body   = parquet_buffer.getvalue()
    )
    ok(f"Parquet DataFrame written → s3://{BUCKET_NAME}/{TEST_KEY_PARQUET}")
except Exception as e:
    fail(f"Parquet write failed: {e}")

# ─────────────────────────────────────────────────────────────
# TEST 9: List Objects in Bucket
# ─────────────────────────────────────────────────────────────
print_header("TEST 9: List Objects in Bucket")
try:
    objects  = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=TEST_PREFIX)
    contents = objects.get("Contents", [])
    if contents:
        ok(f"Found {len(contents)} object(s) under '{TEST_PREFIX}/':")
        for obj in contents:
            print(f"     - {obj['Key']} ({obj['Size']} bytes)")
    else:
        warn("No objects found — previous write tests may have failed")
except Exception as e:
    fail(f"List objects failed: {e}")

# ─────────────────────────────────────────────────────────────
# TEST 10: Cleanup Test Files
# ─────────────────────────────────────────────────────────────
print_header("TEST 10: Cleanup Test Files")
keys_to_delete = [TEST_KEY_TXT, TEST_KEY_CSV, TEST_KEY_PARQUET]
try:
    s3_client.delete_objects(
        Bucket = BUCKET_NAME,
        Delete = {"Objects": [{"Key": k} for k in keys_to_delete]}
    )
    ok(f"Deleted {len(keys_to_delete)} test file(s)")
except Exception as e:
    warn(f"Cleanup failed (manual delete may be needed): {e}")

# ─────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────
print_header("ALL TESTS COMPLETE")
print("  If all ✅ above — MinIO is fully operational from Databricks")
print("  Fix any ❌ before running your main pipeline\n")
